# 02 · Delayed-electron pile-up walk-through

            This is the canonical end-to-end DE notebook. It reproduces the contents of
            the legacy `DEMO.ipynb` / `Run_test.ipynb` on top of the refactored
            :class:`relics_de_sim.Pipeline`.

            Stages:

            1. Muon tracks  → dense interpolation
            2. Delayed electrons (power-law dt + Gaussian xy)
            3. Pile-up grouping (n=2..7)
            4. S2 pattern simulation (light, pe, area)
            5. CNN position reconstruction
            6. Reconstructed-position S2 pattern
            7. Pattern likelihood + space-time correlation

In [ ]:
# Auto-discover the package even if the notebook is launched from outside the repo.
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 110

In [ ]:
from relics_de_sim import DESimConfig, Pipeline
            cfg = DESimConfig.from_yaml('configs/de_sim.yaml')
            print('dead_time_s:', cfg.simulation.dead_time_s)
            print('pile_up_orders:', cfg.simulation.pile_up_orders)

## Stage 1 – Load muons

In [ ]:
from pathlib import Path
            files_per_batch = 5
            muon_files = [Path(cfg.paths.muon_track_dir) / f'muon_track.{i}.npy'
                          for i in range(files_per_batch)]
            pipe = Pipeline(cfg, rng=np.random.default_rng(42))
            pipe.load_muon_tracks(muon_files)
            print(f'simulated time range: {pipe.time_range_s/3600:.2f} h')
            print(f'dead-time ratio (DE survival fraction): we estimate per-config')

## Stage 2 – Delayed electrons

In [ ]:
pipe.simulate_delayed_electrons()
            n_de = len(pipe.delayed_electron)
            print(f'{n_de:_} delayed electrons survived the fiducial cut')
            print(f'  rate: {n_de / pipe.time_range_s:.1f} / s')

            fig, axes = plt.subplots(1, 2, figsize=(10, 4))
            axes[0].hist(pipe.delayed_electron['e_time'], bins=60)
            axes[0].set_xlabel('e_time [s]'); axes[0].set_ylabel('count')
            axes[1].scatter(pipe.delayed_electron['xd'][:5000],
                            pipe.delayed_electron['yd'][:5000], s=2, alpha=0.3)
            axes[1].set_xlabel('x [mm]'); axes[1].set_ylabel('y [mm]'); axes[1].set_aspect('equal')
            axes[0].set_title('DE arrival times')
            axes[1].set_title('first 5k DE positions')
            plt.tight_layout(); plt.show()

## Stage 3 – Pile-up grouping (n = 2..7)

In [ ]:
pipe.simulate_pile_up_patterns()
            for n, res in zip(pipe.pile_up_orders, pipe.pile_up_results):
                print(f'  n={n}: {len(res.pe_by_area)} surviving events')

## Stage 4-5 – Position reconstruction + scoring

In [ ]:
try:
                pipe.recon_position_de()
                pipe.score()
                for n, coef, st in zip(pipe.pile_up_orders, pipe.pile_up_pattern_coef, pipe.pile_up_st_cor):
                    if len(coef):
                        print(f'  n={n}: <pattern_coef>={coef.mean():.2f}, <st_cor>={st.mean():.2e}')
            except ImportError as exc:
                print('Skipping CNN inference (torch unavailable):', exc)

## Stage 6 – Save output

            ``Pipeline.save`` writes the standardised ``de_result.npz`` + ``meta.json``
            (see :mod:`relics_de_sim.io`).

In [ ]:
out = pipe.save('outputs/de_sim/notebook_demo')
            print('wrote', out)